# Change detection in optical satellite time series - Tutorial

<div align="center">
  
  <!-- Institution Logos -->
  <img src="https://www.umr-lastig.fr/static/lastig_1920_EN-5022e6685c1e4b6d7e84d2a08667be4e.png" height="60" alt="LASTIG">
  &nbsp;&nbsp;&nbsp;&nbsp;
  <img src="https://iadf-school.org/wp-content/uploads/2022/09/logo_iadfschool.png" height="60" alt="IADF School">
  &nbsp;&nbsp;&nbsp;&nbsp;
  <img src="https://design-system.ign.fr/img/styleguide/sg_logos/IGN_logo_RVB.png" height="60" alt="IGN">
  
  <br><br>
  
  <!-- Colab Badge -->
  [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ElliotVincent/iadf_tutorial/blob/main/tutorial_2.ipynb)
  
  
</div>

---

## **Tutorial 2: Training and Evaluating bi- and multi-temporal change detection models**

**Welcome to Tutorial 2!** In this hands-on session, you will learn how to train and evaluate bi- and multi-temporal change detection models for optical satellite image time series (SITS).

### What You'll Learn
- Working with a toy dataset of Planet multispectral imagery
-
- Train a semantic segmentation model
- Evaluating model performance in a post-classification framework
- Practical implementation in Google Colab

### Prerequisites
- Basic knowledge of Python and deep learning
- Familiarity with remote sensing concepts
- Google account for Colab access

---

## Introduction

This tutorial demonstrates how to perform change detection in optical satellite time series using Python.  
We will use a sample dataset of satellite images and apply various techniques to identify changes over time.  
The sample dataset is extracted from DynamicEarthNet [1] and focuses on 3 AoIs.  
The map below shows the location of the 3 AoIs in the world:  


![Map of the 3 AoIs](https://github.com/ElliotVincent/iadf_tutorial/blob/main/aois_location.png?raw=1)


In [ ]:
# Import necessary libraries
import copy
import matplotlib.pyplot as plt
import numpy as np
import random
import rasterio
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
# !pip install segmentation_models_pytorch
import segmentation_models_pytorch as smp
from segmentation_models_pytorch.base import SegmentationModel

## 1. Playing with Satellite Image Time Series

### 1.1. Loading and Visualizing the Dataset

In [ ]:
DYNAMICEARTHNET_CLASSES = {
    0: {"label": "other",                     "color": (128, 128, 128)},  # grey
    1: {"label": "vegetation",                "color": ( 34, 139,  34)},  # green
    2: {"label": "water",                     "color": ( 31, 119, 180)},  # blue
}

DATES = [f"{yyyy}-{mm:02d}-01" for yyyy in range(2018, 2020) for mm in range(1, 13)]

# Utility function to perform percentile stretching on the image for better visualization
def percentile_stretch(img, low=2, high=98):
    lo, hi = np.percentile(img, (low, high))
    img_clipped = np.clip(img, lo, hi)
    return ((img_clipped - lo) / (hi - lo) * 255).astype(np.uint8)

# Function to plot a single image
def plot_image(image):
    rgb_img = image[:3]
    rgb_img = rgb_img.transpose(1, 2, 0)
    rgb_img = percentile_stretch(rgb_img)
    plt.imshow(rgb_img)
    plt.axis('off')

# Function to plot a single label
def plot_labels(label):
    rgb_label = np.zeros((label.shape[0], label.shape[1], 3), dtype=np.uint8)
    for i in range(3):
        rgb_label[label == i] = DYNAMICEARTHNET_CLASSES[i]["color"]
    plt.imshow(rgb_label)
    plt.axis('off')

# Function to plot a time series of images and labels
def plot_time_series(images, labels, num_images=10):
    num_images = min(num_images, len(images))
    fig, axes = plt.subplots(3, num_images, figsize=(int(4*num_images / 3), 4))
    for i in range(num_images):
        # Plot the image
        rgb_img = images[i, :3].transpose(1, 2, 0)
        rgb_img = percentile_stretch(rgb_img)
        axes[0, i].set_title(DATES[i], fontsize=10)
        axes[0, i].imshow(rgb_img)
        axes[0, i].axis('off')

        # Plot the semantic maps
        rgb_label = np.zeros((labels[i].shape[0], labels[i].shape[1], 3), dtype=np.uint8)
        for j in range(3):
            rgb_label[labels[i] == j] = DYNAMICEARTHNET_CLASSES[j]["color"]
        axes[1, i].imshow(rgb_label)
        axes[1, i].axis('off')

        # Plot binary change maps
        if i > 0:
            change_map = (labels[i] != labels[i-1]).astype(np.uint8) * 255
            axes[2, i].imshow(change_map, cmap='gray')
            axes[2, i].axis('off')
        else:
            axes[2, i].axis('off')
    plt.tight_layout()
    plt.show()

def load_time_series(path):
    return np.stack([rasterio.open(f"{path}/{date}.tif").read() for date in DATES], axis=0)[:, [2, 1, 0, 3]]

def load_labels(path):
    labels = np.load(f'{path}.npy')
    labels = np.where(labels == 2, 1, labels)
    labels = np.where(labels == 3, 0, labels)
    labels = np.where(labels == 4, 0, labels)
    labels = np.where(labels == 5, 2, labels)
    labels = np.where(labels == 6, 0, labels)
    return labels

We first load the sample dataset and visualize the images for each AoI.  
Satellite image time series (SITS) are stored as T x C x H x W tensors, where T is the number of time steps, C is the number of channels (spectral bands), H is the height, and W is the width of the image.  
In our case, we have 3 AoIs with T=24 (2 years of monthly images between January 2018 and December 2019), C=4 (RGB + NIR), H=1024, and W=1024.  
The spatial resolution of the images is roughly 3 meters per pixel.
DinamicEarthNet dataset is annotated with per-month semantic masks, with 7 land-cover classes: Impervious surface, Agriculture, Forest, Bare soil, Wetlands, Water, Snow/Ice.  
In this tutorial, we simplify the problem by grouping them in 3 classes:
- 0: Other (Impervious surface + Bare soil + Wetlands + Snow/Ice)
- 1: Vegetation (Agriculture + Forest)
- 2: Water (Water)

In [ ]:
!gdown 12R7wzxLnyp-jAQ_97uUqlHd8AjEyTQg5
!gdown 1q4Kx2LnId-eTPW8WZ1yJWwYujlQ2CZnj
!unzip -qq "/content/data.zip"
!unzip -qq "/content/labels.zip"

In [ ]:
aoi_name = '3998_3016_13'

print(f"Loading images and labels...")
images = load_time_series(aoi_name)
labels = load_labels(aoi_name)
plot_time_series(images, labels, num_images=6)

Let's check the shape of each SITS tensor.

In [ ]:
(labels[:-1, 768:, :512] != labels[1:, 768:, :512]).astype(int).sum()

In [ ]:
print(f"Images shape: {images.shape}, labels shape: {labels.shape}")

In [ ]:
class SITSDataset(torch.utils.data.Dataset):
    def __init__(self, data, labels, split, norm=True, num_classes=3, augment=False):
        # Load the dataset once in memory
        self.data = data
        self.labels = labels
        self.norm = norm
        # Dataset statistics for normalization, from DynamicEarthNet paper
        self.mean = torch.tensor([1042.59, 915.62, 671.26, 2605.21])
        self.std = torch.tensor([957.96, 715.55, 596.94, 1059.90])
        self.num_classes = num_classes
        self.ij_lists = {"train":[[i, j] for i in range(16) for j in range(16) if j >= 8 or i < 8],
                         "val": [[i, j] for i in range(16) for j in range(16) if i >= 12 and j < 8],
                         "test": [[i, j] for i in range(16) for j in range(16) if i >= 8 and i < 12 and j < 8]}[split]
        self.split = split
        self.augment = augment

    def __len__(self):
        return len(self.ij_lists)

    def __getitem__(self, idx):
        i, j = self.ij_lists[idx]
        x, y = i*64, j*64
        if self.split == 'train' and self.augment:
          # random crop
          x = min(x + random.randint(0, 63), 7 * 64) if j < 8 else min(x + random.randint(0, 63), 15 * 64)
          y = min(y + random.randint(0, 63), 15 * 64)
        imgs = torch.tensor(self.data[..., x:x+64, y:y+64])
        labs = torch.tensor(self.labels[..., x:x+64, y:y+64])
        if self.norm:
            imgs = (imgs - self.mean[:, None, None]) / self.std[:, None, None]
        if self.augment:
            # random rotation
            rot = random.randint(0, 3)
            imgs = torch.rot90(imgs, rot, [2, 3])
            labs = torch.rot90(labs, rot, [1, 2])
            # random flip
            flip = random.randint(0, 1)
            if flip == 1:
                imgs = torch.flip(imgs, [2])
                labs = torch.flip(labs, [1])
        return imgs, labs

In [ ]:
train_set = SITSDataset(data=images,
                        labels=labels,
                        split='train', norm=True, num_classes=3, augment=True)

val_set = SITSDataset(data=images,
                      labels=labels,
                      split='val', norm=True, num_classes=3)

test_set = SITSDataset(data=images,
                      labels=labels,
                      split='test', norm=True, num_classes=3)

print(f"Train dataset length: {len(train_set)}")
print(f"Validation dataset length: {len(val_set)}")
print(f"Test dataset length: {len(test_set)}")

In [ ]:
def new_forward(self, x, return_features=False, sem_features=None):
    features = self.encoder(x)
    if sem_features is not None:
        sem1, sem2 = sem_features
        for (i,feat) in enumerate(features):
            if i>0:
                features[i] = feat + sem1[i] - sem2[i]
    decoder_output = self.decoder(features)
    masks = self.segmentation_head(decoder_output)
    if return_features:
        return masks, features
    else:
        return masks

SegmentationModel.forward = new_forward

class DualUnet(nn.Module):
    def __init__(self, backbone="mobilenet_v2", in_channels=4, num_classes=3):
        super(DualUnet, self).__init__()
        self.in_channels = in_channels
        self.change_model = smp.Unet(encoder_name=backbone, encoder_weights="imagenet" , in_channels=in_channels * 2, classes=2)
        self.seg_model = smp.Unet(encoder_name=backbone, encoder_weights="imagenet", in_channels=in_channels, classes=num_classes)

    def forward(self, x):
        y1, f1 = self.seg_model(x[0], return_features=True)
        y2, f2 = self.seg_model(x[1], return_features=True)
        sem_preds = (y1, y2)
        change_pred = self.change_model(torch.cat([x[0], x[1]], dim=1), sem_features=(f1, f2))
        return change_pred, sem_preds

model = DualUnet()
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
def compute_metrics(confusion_matrix):
    # Compute overall accuracy
    overall_accuracy = torch.trace(confusion_matrix) / torch.sum(confusion_matrix)

    # Compute mean IoU
    intersection = torch.diagonal(confusion_matrix)
    union = torch.sum(confusion_matrix, dim=1) + torch.sum(confusion_matrix, dim=0) - intersection
    mean_iou = torch.mean(intersection / union)

    return overall_accuracy, mean_iou
import torchvision

def training_step(batch, model, criterion, optimizer, device):
    imgs, labs = batch
    imgs = imgs.to(device)
    labs = labs.to(device)
    # imgs of shape (B, T, C, H, W)
    optimizer.zero_grad()
    temp_indices = torch.tensor([random.sample(range(imgs.size(1)), 2) for _ in range(imgs.size(0))], device=imgs.device)  # (B, 2)
    batch_idx = torch.arange(imgs.size(0), device=imgs.device)[:, None]
    imgs, labs = imgs[batch_idx, temp_indices], labs[batch_idx, temp_indices]
    change_logits, sem_logits = model([imgs[:, 0], imgs[:, 1]])
    change_gt = nn.functional.one_hot((labs[:, 0] != labs[:, 1]).long(), num_classes=2).permute(0, 3, 1, 2).float()
    loss_change = torchvision.ops.sigmoid_focal_loss(change_logits, change_gt, reduction='mean')
    loss_sem = (criterion(sem_logits[0], labs[:, 0].long()) + criterion(sem_logits[1], labs[:, 1].long())) / 2
    loss = loss_change + loss_sem
    loss.backward()
    optimizer.step()
    return loss


def validation_step(batch, model, criterion, device):
    imgs, labs = batch
    imgs = imgs.to(device).squeeze(0)
    labs = labs.to(device).squeeze(0)
    # imgs of shape (1, T, C, H, W)
    imgs = torch.stack([imgs[:-1], imgs[1:]], dim=1)
    change_logits, sem_logits = model([imgs[:, 0], imgs[:, 1]])
    change_gt = nn.functional.one_hot((labs[:-1] != labs[1:]).long(), num_classes=2).permute(0, 3, 1, 2).float()
    loss_change = torchvision.ops.sigmoid_focal_loss(change_logits, change_gt, reduction='mean')
    loss_sem = (criterion(sem_logits[0], labs[:-1].long()) + criterion(sem_logits[1], labs[1:].long())) / 2
    loss = loss_change + loss_sem
    pred_change = torch.argmax(change_logits, dim=1)
    pred_sem = torch.cat([torch.argmax(sem_logits[0], dim=1)[0][None], torch.argmax(sem_logits[1], dim=1)], dim=0)
    return loss, pred_change, pred_sem, (labs[:-1] != labs[1:]).long(), labs.long()

In [ ]:
train_loader = DataLoader(train_set, batch_size=16, shuffle=True)
val_loader = DataLoader(val_set, batch_size=1, shuffle=False)
model = DualUnet()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
num_epochs = 50
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
device = 'cpu'
model.to(device)

logs = {"train_loss": [], "val_loss": [], "val_acc": [], "val_miou": [], "val_iou_change": [], "learning_rate": []}
best_val_miou_change = 0.0
best_model_change = None
best_model_val_conf_matrix = None
best_model_val_conf_matrix_change = None

for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")
    model.train()
    avg_train_loss = 0.0
    for batch in train_loader:
        loss = training_step(batch, model, criterion, optimizer, device)
        avg_train_loss += loss.item()
    avg_train_loss /= len(train_loader)
    print(f"   Average Training Loss: {avg_train_loss:.4f}")

    model.eval()
    avg_val_loss = 0.0
    val_conf_matrix = torch.zeros(3, 3, dtype=torch.long, device=device)
    val_conf_matrix_change = torch.zeros(2, 2, dtype=torch.long, device=device)
    with torch.no_grad():
        for batch in val_loader:
            loss, pred_change, pred_sem, labs_change, labs_sem = validation_step(batch, model, criterion, device)
            avg_val_loss += loss.item()
            val_conf_matrix += torch.bincount(pred_sem.flatten().long() * 3 + labs_sem.flatten().long(), minlength=9).reshape(3, 3)
            val_conf_matrix_change += torch.bincount(pred_change.flatten().long() * 2 + labs_change.flatten().long(), minlength=4).reshape(2, 2)
    avg_val_loss /= len(val_loader)
    acc, miou = compute_metrics(val_conf_matrix)
    change_iou = val_conf_matrix_change[1, 1] / (val_conf_matrix_change[1, 1] + val_conf_matrix_change[1, 0] + val_conf_matrix_change[0, 1])
    print(f"   Average Validation Loss: {avg_val_loss:.4f}")
    print(f"   Validation Accuracy: {acc*100:.2f}%, mIoU: {miou*100:.2f}%")
    print(f"   Change IoU: {change_iou*100:.2f}%")
    logs["train_loss"].append(avg_train_loss)
    logs["val_loss"].append(avg_val_loss)
    logs["val_acc"].append(acc.item())
    logs["val_miou"].append(miou.item())
    logs["val_iou_change"].append(change_iou.item())
    if change_iou.item() > best_val_miou_change:
        best_val_miou_change = change_iou.item()
        best_model_change = copy.deepcopy(model)
        best_model_val_conf_matrix = val_conf_matrix.clone()
        best_model_val_conf_matrix_change = val_conf_matrix_change.clone()
    logs["learning_rate"].append(optimizer.param_groups[0]["lr"])
    scheduler.step()

In [ ]:
# Visualize the trainng curves
plt.figure(figsize=(12, 6))
plt.plot(logs["train_loss"], label="Train Loss")
plt.plot(logs["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Value")
plt.title("Losses")
plt.legend()
plt.show()

# plt.figure(figsize=(12, 6))
# plt.plot(logs["learning_rate"], label="Learning Rate")
# plt.xlabel("Epoch")
# plt.ylabel("Value")
# plt.title("Learning Rate Schedule")
# plt.legend()
# plt.show()

plt.figure(figsize=(12, 6))
plt.plot(logs["val_acc"], label="Validation Accuracy")
plt.plot(logs["val_miou"], label="Validation mIoU")
plt.plot(logs["val_iou_change"], label="Validation Change IoU")
plt.xlabel("Epoch")
plt.ylabel("Value")
plt.title("Validation Metrics")
plt.legend()
plt.show()

In [ ]:
# plot val_confusion matrix
plt.figure(figsize=(8, 6))
# normalize the confusion matrix for better visualization
val_conf_matrix_normalized = best_model_val_conf_matrix.float() / best_model_val_conf_matrix.sum(dim=0, keepdim=True) * 100
sns.heatmap(val_conf_matrix_normalized.cpu().numpy(), annot=True, fmt='.1f', cmap='Blues', xticklabels=[DYNAMICEARTHNET_CLASSES[i]["label"] for i in range(3)], yticklabels=[DYNAMICEARTHNET_CLASSES[i]["label"] for i in range(3)])
plt.ylabel('Predicted Labels')
plt.xlabel('True Labels')
plt.title('Validation Confusion Matrix')
plt.show()

In [ ]:
# plot val_confusion matrix
plt.figure(figsize=(8, 6))
# normalize the confusion matrix for better visualization
val_conf_matrix_normalized = best_model_val_conf_matrix_change.float() / best_model_val_conf_matrix_change.sum(dim=0, keepdim=True) * 100
sns.heatmap(val_conf_matrix_normalized.cpu().numpy(), annot=True, fmt='.1f', cmap='Blues', xticklabels=["No change", "Change"], yticklabels=["No change", "Change"])
plt.ylabel('Predicted Labels')
plt.xlabel('True Labels')
plt.title('Validation Confusion Matrix')
plt.show()

We have trained a simple Unet model for semantic segmentation on our dataset, considering each frame independently.
We have verified that our trained model achieves correct predictions on the validation set, both quantitatively and qualitatively.  

Now, let's evaluate our model on the change detection task on the test set.

In [ ]:
canva_lab = torch.zeros(24, 256, 512).long()
canva_pred = torch.zeros(24, 256, 512).long()
canva_lab_change = torch.zeros(23, 256, 512).long()
canva_pred_change = torch.zeros(23, 256, 512).long()
ij_list = [[i, j] for i in range(4) for j in range(8)]
best_model_change.eval()
for k in range(len(test_set)):
    imgs, labs = test_set[k]
    imgs = torch.stack([imgs[:-1], imgs[1:]], dim=1).to(device)
    with torch.no_grad():
        change_logits, sem_logits = best_model_change([imgs[:, 0], imgs[:, 1]])
    pred_change = torch.argmax(change_logits, dim=1).detach().cpu()
    pred_sem = torch.cat([torch.argmax(sem_logits[0], dim=1)[0][None], torch.argmax(sem_logits[1], dim=1)], dim=0).detach().cpu()
    labs = labs.detach().cpu()
    labs_change = (labs[:-1] != labs[1:]).long()
    i, j = ij_list[k]
    canva_lab[:, i*64:(i+1)*64, j*64:(j+1)*64] = labs
    canva_pred[:, i*64:(i+1)*64, j*64:(j+1)*64] = pred_sem
    canva_lab_change[:, i*64:(i+1)*64, j*64:(j+1)*64] = labs_change
    canva_pred_change[:, i*64:(i+1)*64, j*64:(j+1)*64] = pred_change

In [ ]:
test_confusion_matrix = torch.bincount(canva_pred.flatten().long() * 3 + canva_lab.flatten().long(), minlength=9).reshape(3, 3)
# plot val_confusion matrix
plt.figure(figsize=(8, 6))
# normalize the confusion matrix for better visualization
test_confusion_matrix_normalized = test_confusion_matrix.float() / test_confusion_matrix.sum(dim=0, keepdim=True) * 100
sns.heatmap(test_confusion_matrix_normalized.cpu().numpy(), annot=True, fmt='.1f', cmap='Blues', xticklabels=[DYNAMICEARTHNET_CLASSES[i]["label"] for i in range(3)], yticklabels=[DYNAMICEARTHNET_CLASSES[i]["label"] for i in range(3)])
plt.ylabel('Predicted Labels')
plt.xlabel('True Labels')
plt.title('Test Confusion Matrix')
plt.show()

In [ ]:
test_confusion_matrix_change = torch.bincount(canva_pred_change.flatten().long() * 2 + canva_lab_change.flatten().long(), minlength=4).reshape(2, 2)
# plot val_confusion matrix
plt.figure(figsize=(8, 6))
# normalize the confusion matrix for better visualization
test_confusion_matrix_change_normalized = test_confusion_matrix_change.float() / test_confusion_matrix_change.sum(dim=0, keepdim=True) * 100
sns.heatmap(test_confusion_matrix_change_normalized.cpu().numpy(), annot=True, fmt='.1f', cmap='Blues', xticklabels=['No Change', 'Change'], yticklabels=['No Change', 'Change'])
plt.ylabel('Predicted Labels')
plt.xlabel('True Labels')
plt.title('Test Confusion Matrix for Change Detection')
plt.show()

In [ ]:
plt.figure(figsize=(15, 5))
for time_step in range(6):
    plt.subplot(3, 6, time_step + 1)
    plt.title(f"Time Step {time_step}")
    plot_image(images[time_step, :, 512:768:, :512])
    if time_step > 0:
        plt.subplot(3, 6, time_step + 7)
        plt.title(f"Ground Truth")
        plt.imshow(canva_lab_change[time_step].numpy(), cmap='gray')
        plt.axis('off')
        plt.subplot(3, 6, time_step + 13)
        plt.title(f"Prediction")
        plt.imshow(canva_pred_change[time_step].numpy(), cmap='gray')
        plt.axis('off')
plt.show()

Looking at the predicted binary change mask, compared to the ground truth, we can observe two things:
1. Predicted change maps are very noisy, with a significant amount of false positives.  
2. Some real changes are missed by the model, leading to false negatives.  
Overall, change performance are low, as shown by the change IoU and the false positive rate.

In [ ]:
change_iou = test_confusion_matrix_change[1, 1].float() / (test_confusion_matrix_change[1, 1].float() + test_confusion_matrix_change[1, 0].float() + test_confusion_matrix_change[0, 1].float())
false_positive_rate = test_confusion_matrix_change[1, 0].float() / (test_confusion_matrix_change[1, 0].float() + test_confusion_matrix_change[0, 0].float())
print(f"Change IoU: {change_iou.item() * 100:.1f}")
print(f"False Positive Rate: {false_positive_rate.item() * 100:.1f}")

In [ ]:
class PositionalEncoder(nn.Module):
    def __init__(self, d, T=730, repeat=None, offset=0):
        super(PositionalEncoder, self).__init__()
        self.d = d
        self.T = T
        self.repeat = repeat
        self.denom = torch.pow(
            T, 2 * torch.div(torch.arange(offset, offset + d).float(), 2, rounding_mode='floor') / (d+offset)
        )
        self.updated_location = False

    def forward(self, batch_positions):
        if not self.updated_location:
            self.denom = self.denom.to(batch_positions.device)
            self.updated_location = True
        sinusoid_table = (
            batch_positions[:, :, None] / self.denom[None, None, :]
        )  # B x T x C
        sinusoid_table[:, :, 0::2] = torch.sin(sinusoid_table[:, :, 0::2])  # dim 2i
        sinusoid_table[:, :, 1::2] = torch.cos(sinusoid_table[:, :, 1::2])  # dim 2i+1

        if self.repeat is not None:
            sinusoid_table = torch.cat(
                [sinusoid_table for _ in range(self.repeat)], dim=-1
            )

        return sinusoid_table


class TemporallySharedBlock(nn.Module):
    """
    Helper module for convolutional encoding blocks that are shared across a sequence.
    This module adds the self.smart_forward() method the the block.
    smart_forward will combine the batch and temporal dimension of an input tensor
    if it is 5-D and apply the shared convolutions to all the (batch x temp) positions.
    """

    def __init__(self, pad_value=None):
        super(TemporallySharedBlock, self).__init__()
        self.out_shape = None
        self.pad_value = pad_value

    def smart_forward(self, input):
        if len(input.shape) == 4:
            return self.forward(input)
        else:
            b, t, c, h, w = input.shape

            if self.pad_value is not None:
                dummy = torch.zeros(input.shape, device=input.device).float()
                self.out_shape = self.forward(dummy.view(b * t, c, h, w)).shape

            out = input.view(b * t, c, h, w)
            if self.pad_value is not None:
                pad_mask = (out == self.pad_value).all(dim=-1).all(dim=-1).all(dim=-1)
                if pad_mask.any():
                    temp = (
                        torch.ones(
                            self.out_shape, device=input.device, requires_grad=False
                        )
                        * self.pad_value
                    )
                    temp[~pad_mask] = self.forward(out[~pad_mask])
                    out = temp
                else:
                    out = self.forward(out)
            else:
                out = self.forward(out)
            _, c, h, w = out.shape
            out = out.view(b, t, c, h, w)
            return out


class ConvLayer(nn.Module):
    def __init__(
        self,
        nkernels,
        norm="batch",
        k=3,
        s=1,
        p=1,
        n_groups=4,
        last_relu=True,
        padding_mode="reflect",
    ):
        super(ConvLayer, self).__init__()
        layers = []
        if norm == "batch":
            nl = nn.BatchNorm2d
        elif norm == "instance":
            nl = nn.InstanceNorm2d
        elif norm == "group":
            nl = lambda num_feats: nn.GroupNorm(
                num_channels=num_feats,
                num_groups=n_groups,
            )
        else:
            nl = None
        for i in range(len(nkernels) - 1):
            layers.append(
                nn.Conv2d(
                    in_channels=nkernels[i],
                    out_channels=nkernels[i + 1],
                    kernel_size=k,
                    padding=p,
                    stride=s,
                    padding_mode=padding_mode,
                )
            )
            if nl is not None:
                layers.append(nl(nkernels[i + 1]))

            if last_relu:
                layers.append(nn.ReLU())
            elif i < len(nkernels) - 2:
                layers.append(nn.ReLU())
        self.conv = nn.Sequential(*layers)

    def forward(self, input):
        return self.conv(input)


class ConvBlock(TemporallySharedBlock):
    def __init__(
        self,
        nkernels,
        pad_value=None,
        norm="batch",
        last_relu=True,
        padding_mode="reflect",
    ):
        super(ConvBlock, self).__init__(pad_value=pad_value)
        self.conv = ConvLayer(
            nkernels=nkernels,
            norm=norm,
            last_relu=last_relu,
            padding_mode=padding_mode,
        )

    def forward(self, input):
        return self.conv(input)


class DownConvBlock(TemporallySharedBlock):
    def __init__(
        self,
        d_in,
        d_out,
        k,
        s,
        p,
        pad_value=None,
        norm="batch",
        padding_mode="reflect"
    ):
        super(DownConvBlock, self).__init__(pad_value=pad_value)
        self.down = ConvLayer(
            nkernels=[d_in, d_in],
            norm=norm,
            k=k,
            s=s,
            p=p,
            padding_mode=padding_mode,
        )
        self.conv1 = ConvLayer(
            nkernels=[d_in, d_out],
            norm=norm,
            padding_mode=padding_mode,
        )
        self.conv2 = ConvLayer(
            nkernels=[d_out, d_out],
            norm=norm,
            padding_mode=padding_mode,
        )

    def forward(self, input):
        out = self.down(input)
        out = self.conv1(out)
        out = out + self.conv2(out)
        return out


class UpConvBlock(nn.Module):
    def __init__(
        self, d_in, d_out, k, s, p, norm="batch", d_skip=None, padding_mode="reflect"
    ):
        super(UpConvBlock, self).__init__()
        d = d_out if d_skip is None else d_skip
        self.skip_conv = nn.Sequential(
            nn.Conv2d(in_channels=d, out_channels=d, kernel_size=1),
            nn.ReLU(),
        )
        self.up = nn.Sequential(
            nn.ConvTranspose2d(
                in_channels=d_in, out_channels=d_out, kernel_size=k, stride=s, padding=p
            ),
            nn.ReLU(),
        )
        self.conv1 = ConvLayer(
            nkernels=[d_out + d, d_out], norm=norm, padding_mode=padding_mode
        )
        self.conv2 = ConvLayer(
            nkernels=[d_out, d_out], norm=norm, padding_mode=padding_mode
        )

    def forward(self, input, skip):
        bs, seq_len, c, h, w = input.size()
        input = input.contiguous().view(bs * seq_len, c, h, w)
        out = self.up(input)
        _, _, cs, hs, ws = skip.size()
        skip = skip.view(bs * seq_len, cs, hs, ws)
        out = torch.cat([out, self.skip_conv(skip)], dim=1)
        out = self.conv1(out)
        out = out + self.conv2(out)
        _, c, h, w = out.size()
        out = out.view(bs, seq_len, c, h, w)
        return out


class MultiLTAE(nn.Module):
    def __init__(
        self,
        in_channels=128,
        n_head=16,
        d_k=4,
        dropout=0.2,
        T=730,
        offset=0,
        return_att=False,
        positional_encoding=True
    ):
        """
        Lightweight Temporal Attention Encoder (L-TAE) for image time series.
        Attention-based sequence encoding that maps a sequence of images to a single feature map.
        A shared L-TAE is applied to all pixel positions of the image sequence.
        Args:
            in_channels (int): Number of channels of the input embeddings.
            n_head (int): Number of attention heads.
            d_k (int): Dimension of the key and query vectors.
            mlp (List[int]): Widths of the layers of the MLP that processes the concatenated outputs of the attention heads.
            dropout (float): dropout
            d_model (int, optional): If specified, the input tensors will first processed by a fully connected layer
                to project them into a feature space of dimension d_model.
            T (int): Period to use for the positional encoding.
            return_att (bool): If true, the module returns the attention masks along with the embeddings (default False)
            positional_encoding (bool): If False, no positional encoding is used (default True).
        """
        super(MultiLTAE, self).__init__()
        self.in_channels = in_channels
        self.mlp = [in_channels, in_channels]
        self.return_att = return_att
        self.n_head = n_head
        self.d_model = in_channels
        assert self.mlp[0] == self.d_model

        if positional_encoding:
            self.positional_encoder = PositionalEncoder(
                self.d_model // n_head, T=T, repeat=n_head, offset=offset
            )
        else:
            self.positional_encoder = None

        self.attention_heads = MultiHeadAttention(
            n_head=n_head, d_k=d_k, d_in=self.d_model
        )
        self.in_norm = nn.GroupNorm(
            num_groups=n_head,
            num_channels=self.in_channels,
        )
        self.out_norm = nn.GroupNorm(
            num_groups=n_head,
            num_channels=self.mlp[-1],
        )

        layers = []
        for i in range(len(self.mlp) - 1):
            layers.extend(
                [
                    nn.Linear(self.mlp[i], self.mlp[i + 1]),
                    nn.ReLU(),
                ]
            )

        self.mlp = nn.Sequential(*layers)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, batch_positions=None, pad_mask=None):
        sz_b, seq_len, d, h, w = x.shape
        if pad_mask is not None:
            pad_mask = (
                pad_mask.unsqueeze(-1)
                .repeat((1, 1, h))
                .unsqueeze(-1)
                .repeat((1, 1, 1, w))
            )  # BxTxHxW
            pad_mask = (
                pad_mask.permute(0, 2, 3, 1).contiguous().view(sz_b * h * w, seq_len)
            )

        out = x.permute(0, 3, 4, 1, 2).contiguous().view(sz_b * h * w, seq_len, d)
        out = self.in_norm(out.permute(0, 2, 1)).permute(0, 2, 1)

        if self.positional_encoder is not None:
            bp = (
                batch_positions.unsqueeze(-1)
                .repeat((1, 1, h))
                .unsqueeze(-1)
                .repeat((1, 1, 1, w))
            )  # BxTxHxW
            bp = bp.permute(0, 2, 3, 1).contiguous().view(sz_b * h * w, seq_len)
            out = out + self.positional_encoder(bp)

        out, attn = self.attention_heads(out, pad_mask=pad_mask)  # h x (sz_b*h*w) x t x (d//h), h x (sz_b*h*w) x t x t
        out = out.permute(1, 2, 0, 3).contiguous().view(sz_b * h * w, seq_len, -1)  # Concatenate heads

        out = self.dropout(self.mlp(out.view(sz_b * h * w * seq_len, -1)))
        out = self.out_norm(out) if self.out_norm is not None else out
        out = out.view(sz_b, h, w, seq_len, -1).permute(0, 3, 4, 1, 2)

        attn = attn.view(self.n_head, sz_b, h, w, seq_len, seq_len).permute(
            0, 1, 4, 5, 2, 3
        )  # head x b x t x t x h x w

        if self.return_att:
            return out, attn
        else:
            return out


class MultiHeadAttention(nn.Module):
    """Multi-Head Attention module
    Modified from github.com/jadore801120/attention-is-all-you-need-pytorch
    """

    def __init__(self, n_head, d_k, d_in):
        super().__init__()
        self.n_head = n_head
        self.d_k = d_k
        self.d_in = d_in

        self.fc1_q = nn.Linear(d_in, n_head * d_k)
        nn.init.normal_(self.fc1_q.weight, mean=0, std=np.sqrt(2.0 / (d_k)))

        self.fc1_k = nn.Linear(d_in, n_head * d_k)
        nn.init.normal_(self.fc1_k.weight, mean=0, std=np.sqrt(2.0 / (d_k)))

        self.attention = ScaledDotProductAttention(temperature=np.power(d_k, 0.5))

    def forward(self, v, pad_mask=None):
        d_k, d_in, n_head = self.d_k, self.d_in, self.n_head
        sz_b, seq_len, _ = v.size()

        q = self.fc1_q(v).view(sz_b, seq_len, n_head, d_k)
        q = q.permute(2, 0, 1, 3).contiguous().view(-1, seq_len, d_k).permute(0, 2, 1)  # (n*b) x dk x lk

        k = self.fc1_k(v).view(sz_b, seq_len, n_head, d_k)
        k = k.permute(2, 0, 1, 3).contiguous().view(-1, seq_len, d_k)  # (n*b) x lk x dk

        if pad_mask is not None:
            pad_mask = pad_mask.repeat(
                (n_head, 1)
            )  # replicate pad_mask for each head (nxb) x lk

        v = torch.stack(v.split(v.shape[-1] // n_head, dim=-1)).view(
            n_head * sz_b, seq_len, -1
        )
        output, attn = self.attention(q, k, v, pad_mask=pad_mask)
        attn = attn.view(n_head, sz_b, seq_len, seq_len)
        output = output.view(n_head, sz_b, seq_len, d_in // n_head)
        return output, attn


class ScaledDotProductAttention(nn.Module):
    """Scaled Dot-Product Attention
    Modified from github.com/jadore801120/attention-is-all-you-need-pytorch
    """

    def __init__(self, temperature, attn_dropout=0.1):
        super().__init__()
        self.temperature = temperature
        self.dropout = nn.Dropout(attn_dropout)
        self.softmax = nn.Softmax(dim=2)

    def forward(self, q, k, v, pad_mask=None):
        attn = torch.matmul(k, q)
        attn = attn / self.temperature

        if pad_mask is not None:
            attn = attn.masked_fill(pad_mask.unsqueeze(1), -1e3)

        attn = self.softmax(attn)
        attn = self.dropout(attn)
        output = torch.matmul(attn, v)
        return output, attn


class MultiUTAE(nn.Module):
    def __init__(
        self,
        input_dim,
        num_classes,
        in_features=128,
        str_conv_k=4,
        str_conv_s=2,
        str_conv_p=1,
        agg_mode="att_group",
        encoder_norm="group",
        n_head=16,
        d_k=4,
        pad_value=0,
        padding_mode="reflect",
        T=1000,
        offset=10
    ):
        """
        Multi-temporal U-TAE architecture for multi-stamp spatio-temporal encoding of satellite image time series.
        Args:
            input_dim (int): Number of channels in the input images.
            num_classes (int): Number of classes i.e. number of output channels.
            in_features (int): Feature size at the innermost stage.
            str_conv_k (int): Kernel size of the strided up and down convolutions.
            str_conv_s (int): Stride of the strided up and down convolutions.
            str_conv_p (int): Padding of the strided up and down convolutions.
            agg_mode (str): Aggregation mode for the skip connections. Can either be:
                - att_group (default) : Attention weighted temporal average, using the same
                channel grouping strategy as in the LTAE. The attention masks are bilinearly
                resampled to the resolution of the skipped feature maps.
                - att_mean : Attention weighted temporal average,
                 using the average attention scores across heads for each date.
                - mean : Temporal average excluding padded dates.
            encoder_norm (str): Type of normalisation layer to use in the encoding branch. Can either be:
                - group : GroupNorm (default)
                - batch : BatchNorm
                - instance : InstanceNorm
            n_head (int): Number of heads in LTAE.
            d_k (int): Key-Query space dimension
            pad_value (float): Value used by the dataloader for temporal padding.
            padding_mode (str): Spatial padding strategy for convolutional layers (passed to nn.Conv2d).
        """
        super().__init__()
        self.encoder_widths = [in_features // 2, in_features // 2, in_features // 2, in_features]
        self.decoder_widths = [in_features // 4, in_features // 4, in_features // 2, in_features]
        self.n_stages = len(self.encoder_widths)
        self.enc_dim = (
            self.decoder_widths[0] if self.decoder_widths is not None else self.encoder_widths[0]
        )
        self.stack_dim = (
            sum(self.decoder_widths) if self.decoder_widths is not None else sum(self.encoder_widths)
        )
        self.pad_value = pad_value

        if self.decoder_widths is not None:
            assert len(self.encoder_widths) == len(self.decoder_widths)
            assert self.encoder_widths[-1] == self.decoder_widths[-1]
        else:
            self.decoder_widths = self.encoder_widths

        in_conv_kernels = [input_dim] + [self.encoder_widths[0], self.encoder_widths[0]]
        self.in_conv = ConvBlock(
            nkernels=in_conv_kernels,
            pad_value=pad_value,
            norm=encoder_norm,
            padding_mode=padding_mode,
        )
        self.down_blocks = nn.ModuleList(
            DownConvBlock(
                d_in=self.encoder_widths[i],
                d_out=self.encoder_widths[i + 1],
                k=str_conv_k,
                s=str_conv_s,
                p=str_conv_p,
                pad_value=pad_value,
                norm=encoder_norm,
                padding_mode=padding_mode
            )
            for i in range(self.n_stages - 1)
        )
        self.up_blocks = nn.ModuleList(
            UpConvBlock(
                d_in=self.decoder_widths[i],
                d_out=self.decoder_widths[i - 1],
                d_skip=self.encoder_widths[i - 1],
                k=str_conv_k,
                s=str_conv_s,
                p=str_conv_p,
                norm="group",
                padding_mode=padding_mode
            )
            for i in range(self.n_stages - 1, 0, -1)
        )
        self.temporal_encoder = MultiLTAE(
            in_channels=self.encoder_widths[-1],
            n_head=n_head,
            return_att=True,
            d_k=d_k,
            T=T,
            offset=offset
        )
        self.temporal_aggregator = Temporal_Aggregator(mode=agg_mode)
        self.out_conv = ConvBlock(nkernels=[self.decoder_widths[0]] + [in_features // 4, num_classes],
                                  padding_mode=padding_mode,
                                  norm='None')

    def forward(self, batch):
        x = batch["data"]
        batch_positions = batch["positions"]
        batch_size, seq_len, c, h, w = x.size()
        if batch_positions is None:
            batch_positions = torch.tensor(range(1, x.shape[1]+1),
                                           dtype=torch.long,
                                           device=x.device)[None].expand(x.shape[0], -1, -1, -1, -1)

        pad_mask = (
            (x == self.pad_value).all(dim=-1).all(dim=-1).all(dim=-1)
        )  # BxT pad mask
        out = self.in_conv.smart_forward(x)
        feature_maps = [out]

        # SPATIAL ENCODER
        for i in range(self.n_stages - 1):
            out = self.down_blocks[i].smart_forward(feature_maps[-1])
            feature_maps.append(out)

        # TEMPORAL ENCODER
        out, att = self.temporal_encoder(
            feature_maps[-1], batch_positions=batch_positions, pad_mask=pad_mask
        )

        # SPATIAL DECODER
        for i in range(self.n_stages - 1):
            skip = self.temporal_aggregator(
                feature_maps[-(i + 2)], pad_mask=pad_mask, attn_mask=att
            )
            out = self.up_blocks[i](out, skip)

        out = self.out_conv(out.view(batch_size * seq_len, -1, h, w)).view(batch_size, seq_len, -1, h, w)
        return {"logits": out}


class Temporal_Aggregator(nn.Module):
    def __init__(self, mode="mean"):
        super(Temporal_Aggregator, self).__init__()
        self.mode = mode

    def forward(self, x, pad_mask=None, attn_mask=None):
        if pad_mask is not None and pad_mask.any():
            if self.mode == "att_group":
                n_heads, b, t, h, w = attn_mask.shape
                attn = attn_mask.view(n_heads * b, t, h, w)

                if x.shape[-2] > w:
                    attn = nn.Upsample(
                        size=x.shape[-2:], mode="bilinear", align_corners=False
                    )(attn)
                else:
                    attn = nn.AvgPool2d(kernel_size=w // x.shape[-2])(attn)

                attn = attn.view(n_heads, b, t, *x.shape[-2:])
                attn = attn * (~pad_mask).float()[None, :, :, None, None]

                out = torch.stack(x.chunk(n_heads, dim=2))  # hxBxTxC/hxHxW
                out = attn[:, :, :, None, :, :] * out
                out = out.sum(dim=2)  # sum on temporal dim -> hxBxC/hxHxW
                out = torch.cat([group for group in out], dim=1)  # -> BxCxHxW
                return out
            elif self.mode == "att_mean":
                attn = attn_mask.mean(dim=0)  # average over heads -> BxTxHxW
                attn = nn.Upsample(
                    size=x.shape[-2:], mode="bilinear", align_corners=False
                )(attn)
                attn = attn * (~pad_mask).float()[:, :, None, None]
                out = (x * attn[:, :, None, :, :]).sum(dim=1)
                return out
            elif self.mode == "mean":
                out = x * (~pad_mask).float()[:, :, None, None, None]
                out = out.sum(dim=1) / (~pad_mask).sum(dim=1)[:, None, None, None]
                return out
        else:
            if self.mode == "att_group":
                n_heads, b, t, _, h, w = attn_mask.shape
                attn = attn_mask.contiguous().view(n_heads * b * t, t, h, w)
                if x.shape[-2] > w:
                    attn = nn.Upsample(
                        size=x.shape[-2:], mode="bilinear", align_corners=False
                    )(attn)
                else:
                    attn = nn.AvgPool2d(kernel_size=w // x.shape[-2])(attn)
                attn = attn.view(n_heads, b, t, t, *x.shape[-2:])  # hxBxTxTxHxW
                out = torch.stack(x.chunk(n_heads, dim=2))  # hxBxTx(C/h)xHxW
                out = torch.einsum('nbtuhw, nbukhw -> nbtukhw', attn, out)
                out = out.sum(dim=3)  # sum on temporal dim -> hxBxTx(C/h)xHxW
                out = torch.cat([group for group in out], dim=2)  # -> BxTxCxHxW
                return out
            elif self.mode == "att_mean":
                attn = attn_mask.mean(dim=0)  # average over heads -> BxTxHxW
                attn = nn.Upsample(
                    size=x.shape[-2:], mode="bilinear", align_corners=False
                )(attn)
                out = (x * attn[:, :, None, :, :]).sum(dim=1)
                return out
            elif self.mode == "mean":
                return x.mean(dim=1)


def create_mlp(in_ch, out_ch, n_hidden_units, n_layers):
    if n_layers > 0:
        seq = [nn.Linear(in_ch, n_hidden_units), nn.ReLU(True)]
        for _ in range(n_layers - 1):
            seq += [nn.Linear(n_hidden_units, n_hidden_units), nn.ReLU(True)]
        seq += [nn.Linear(n_hidden_units, out_ch)]
    else:
        seq = [nn.Linear(in_ch, out_ch)]
    return nn.Sequential(*seq)

In [ ]:
model = MultiUTAE(input_dim=4, num_classes=3)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
def training_step(batch, model, criterion, optimizer, device):
    imgs, labs = batch
    imgs = imgs.to(device)
    labs = labs.to(device)
    # imgs of shape (B, T, C, H, W)
    optimizer.zero_grad()
    temp_indices = torch.tensor([random.sample(range(imgs.size(1)), 8) for _ in range(imgs.size(0))], device=imgs.device)  # (B, 2)
    temp_indices, _ = torch.sort(temp_indices, dim=1)
    batch_idx = torch.arange(imgs.size(0), device=imgs.device)[:, None]
    imgs, labs, days = imgs[batch_idx, temp_indices], labs[batch_idx, temp_indices], DAYS[None].expand((imgs.size(0), -1))[batch_idx, temp_indices]
    sem_logits = model({"data": imgs, "positions": days})["logits"]
    loss = criterion(sem_logits.reshape(-1, 3, 64, 64), labs.long().reshape(-1, 64, 64))
    loss.backward()
    optimizer.step()
    return loss


def validation_step(batch, model, criterion, device):
    imgs, labs = batch
    imgs = imgs.to(device)
    labs = labs.to(device).squeeze(0)
    # imgs of shape (1, T, C, H, W)
    sem_logits = model({"data": imgs, "positions": DAYS.unsqueeze(0)})["logits"]
    change_gt = nn.functional.one_hot((labs[:-1] != labs[1:]).long(), num_classes=2).permute(0, 3, 1, 2).float()
    loss = criterion(sem_logits.reshape(-1, 3, 64, 64), labs.long().reshape(-1, 64, 64))
    pred_sem = torch.argmax(sem_logits.squeeze(0), dim=1)
    pred_change = (pred_sem[:-1] != pred_sem[1:]).long()
    return loss, pred_change, pred_sem, (labs[:-1] != labs[1:]).long(), labs.long()

In [ ]:
train_loader = DataLoader(train_set, batch_size=16, shuffle=True)
val_loader = DataLoader(val_set, batch_size=1, shuffle=False)
model = MultiUTAE(input_dim=4, num_classes=3)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3)
num_epochs = 50
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
device = 'cpu'
DAYS = torch.tensor([0, 31, 59, 90, 120, 151, 181, 212, 243, 273, 304, 334, 365, 396, 424, 455, 485, 516, 546, 577, 608, 638, 669, 699], device=device)
model.to(device)

logs = {"train_loss": [], "val_loss": [], "val_acc": [], "val_miou": [], "val_iou_change": [], "learning_rate": []}
best_val_iou_change = 0.0
best_model_change = None
best_model_val_conf_matrix = None
best_model_val_conf_matrix_change = None

for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")
    model.train()
    avg_train_loss = 0.0
    for batch in train_loader:
        loss = training_step(batch, model, criterion, optimizer, device)
        avg_train_loss += loss.item()
    avg_train_loss /= len(train_loader)
    print(f"   Average Training Loss: {avg_train_loss:.4f}")

    model.eval()
    avg_val_loss = 0.0
    val_conf_matrix = torch.zeros(3, 3, dtype=torch.long, device=device)
    val_conf_matrix_change = torch.zeros(2, 2, dtype=torch.long, device=device)
    with torch.no_grad():
        for batch in val_loader:
            loss, pred_change, pred_sem, labs_change, labs_sem = validation_step(batch, model, criterion, device)
            avg_val_loss += loss.item()
            val_conf_matrix += torch.bincount(pred_sem.flatten().long() * 3 + labs_sem.flatten().long(), minlength=9).reshape(3, 3)
            val_conf_matrix_change += torch.bincount(pred_change.flatten().long() * 2 + labs_change.flatten().long(), minlength=4).reshape(2, 2)
    avg_val_loss /= len(val_loader)
    acc, miou = compute_metrics(val_conf_matrix)
    iou_change = val_conf_matrix_change[1, 1] / (val_conf_matrix_change[1, 1] + val_conf_matrix_change[1, 0] + val_conf_matrix_change[0, 1])
    print(f"   Average Validation Loss: {avg_val_loss:.4f}")
    print(f"   Validation Accuracy: {acc*100:.2f}%, mIoU: {miou*100:.2f}%")
    print(f"   Change IoU: {iou_change*100:.2f}%")
    logs["train_loss"].append(avg_train_loss)
    logs["val_loss"].append(avg_val_loss)
    logs["val_acc"].append(acc.item())
    logs["val_miou"].append(miou.item())
    logs["val_iou_change"].append(iou_change)
    if iou_change.item() > best_val_iou_change:
        best_val_iou_change = iou_change.item()
        best_model_change = copy.deepcopy(model)
        best_model_val_conf_matrix = val_conf_matrix.clone()
        best_model_val_conf_matrix_change = val_conf_matrix_change.clone()
    logs["learning_rate"].append(optimizer.param_groups[0]["lr"])
    scheduler.step()

### Bibliography

[1] Toker et al. _DynamicEarthNet: Daily Multi-Spectral Satellite Dataset for Semantic Change Segmentation_. CVPR 2022.